In [2]:
from google.colab import drive 
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
# Define datasets we are going to be working with, for readablity
DATASET_PATH = "/content/drive/MyDrive/Datasets/ferret_filtered_jpg_top3"

# extract other things for dataset name
DATASET_NAME = DATASET_PATH.split('/')[-1]
print(f"Working with dataset: {DATASET_NAME}")
ALIGNED_PATH = "/content/drive/MyDrive/Datasets_aligned"

Working with dataset: ferret_filtered_jpg_top3


In [3]:
# have to use those stupid fucking ! things to clone 
# and then cd cause it's not persistant and shit
!git clone https://github.com/alyssaq/face_morpher.git
%cd face_morpher

!mkdir -p data

!curl -LO https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -d shape_predictor_68_face_landmarks.dat.bz2
!mv shape_predictor_68_face_landmarks.dat data/

# and normal export is not going to work as well
import os
os.environ["DLIB_DATA_DIR"] = os.path.abspath("data")

Cloning into 'face_morpher'...
remote: Enumerating objects: 942, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 942 (delta 0), reused 1 (delta 0), pack-reused 937 (from 1)
Receiving objects: 100% (942/942), 75.59 MiB | 13.93 MiB/s, done.
Resolving deltas: 100% (575/575), done.
/content/face_morpher
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 61.0M  100 61.0M    0     0  31.7M      0  0:00:01  0:00:01 --:--:-- 31.6M


In [4]:
%ls

data/      facemorpher/  requirements.txt  setup.cfg
examples/  README.rst    scripts/          setup.py


In [5]:
%%bash
# installing the stuff and settng up the morpher as module
pip install -r requirements.txt
pip install -e .

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=37304294383ec00d0400a7229994b36832a62541cb1ffed301fd2b11ff470679
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt
Obtaining file:///content/face_morpher
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py develop for facemorpher


In [6]:
# python imports block
import subprocess
import sys
from facemorpher import morpher
import os
import glob
import bz2
import requests
import numpy as np
import scipy.ndimage
import PIL.Image
import dlib
from tqdm import tqdm
import torch
import pandas as pd

In [7]:
# alignment functions
# --- FFHQ Standard Image Alignment Transform ---
def image_align(src_file, dst_file, face_landmarks, output_size=256, transform_size=4096, enable_padding=True):
    lm = np.array(face_landmarks)
    lm_eye_left = lm[36:42]
    lm_eye_right = lm[42:48]
    lm_mouth_outer = lm[48:60]

    eye_left = np.mean(lm_eye_left, axis=0)
    eye_right = np.mean(lm_eye_right, axis=0)
    eye_avg = (eye_left + eye_right) * 0.5
    eye_to_eye = eye_right - eye_left
    mouth_left = lm_mouth_outer[0]
    mouth_right = lm_mouth_outer[6]
    mouth_avg = (mouth_left + mouth_right) * 0.5
    eye_to_mouth = mouth_avg - eye_avg

    x = eye_to_eye - np.flipud(eye_to_mouth) * [-1, 1]
    x /= np.hypot(*x)
    x *= max(np.hypot(*eye_to_eye) * 2.0, np.hypot(*eye_to_mouth) * 1.8)
    y = np.flipud(x) * [-1, 1]
    c = eye_avg + eye_to_mouth * 0.1
    quad = np.stack([c - x - y, c - x + y, c + x + y, c + x - y])
    qsize = np.hypot(*x) * 2

    img = PIL.Image.open(src_file).convert('RGB')

    shrink = int(np.floor(qsize / output_size * 0.5))
    if shrink > 1:
        rsize = (int(np.rint(float(img.size[0]) / shrink)), int(np.rint(float(img.size[1]) / shrink)))
        img = img.resize(rsize, PIL.Image.LANCZOS)
        quad /= shrink
        qsize /= shrink

    border = max(int(np.rint(qsize * 0.1)), 3)
    crop = (int(np.floor(min(quad[:, 0]))), int(np.floor(min(quad[:, 1]))),
            int(np.ceil(max(quad[:, 0]))), int(np.ceil(max(quad[:, 1]))))
    crop = (max(crop[0] - border, 0), max(crop[1] - border, 0),
            min(crop[2] + border, img.size[0]), min(crop[3] + border, img.size[1]))
    if crop[2] - crop[0] < img.size[0] or crop[3] - crop[1] < img.size[1]:
        img = img.crop(crop)
        quad -= crop[0:2]

    pad = (int(np.floor(min(quad[:, 0]))), int(np.floor(min(quad[:, 1]))),
           int(np.ceil(max(quad[:, 0]))), int(np.ceil(max(quad[:, 1]))))
    pad = (max(-pad[0] + border, 0), max(-pad[1] + border, 0),
           max(pad[2] - img.size[0] + border, 0), max(pad[3] - img.size[1] + border, 0))
    if enable_padding and max(pad) > border - 4:
        pad = np.maximum(pad, int(np.rint(qsize * 0.3)))
        img = np.pad(np.float32(img), ((pad[1], pad[3]), (pad[0], pad[2]), (0, 0)), 'reflect')
        h, w, _ = img.shape
        y_grid, x_grid, _ = np.ogrid[:h, :w, :1]
        mask = np.maximum(
            1.0 - np.minimum(np.float32(x_grid) / pad[0], np.float32(w - 1 - x_grid) / pad[2]),
            1.0 - np.minimum(np.float32(y_grid) / pad[1], np.float32(h - 1 - y_grid) / pad[3])
        )
        blur = qsize * 0.02
        img += (scipy.ndimage.gaussian_filter(img, [blur, blur, 0]) - img) * np.clip(mask * 3.0 + 1.0, 0.0, 1.0)
        img += (np.median(img, axis=(0, 1)) - img) * np.clip(mask, 0.0, 1.0)
        img = PIL.Image.fromarray(np.uint8(np.clip(np.rint(img), 0, 255)), 'RGB')
        quad += pad[:2]

    img = img.transform((transform_size, transform_size), PIL.Image.QUAD, (quad + 0.5).flatten(), PIL.Image.BILINEAR)
    if output_size < transform_size:
        img = img.resize((output_size, output_size), PIL.Image.LANCZOS)

    img.save(dst_file, 'PNG')

# --- Recursive Dataset Pipeline ---
def align_dataset(input_root, output_root):
    model_path = get_shape_predictor()
    detector = dlib.get_frontal_face_detector()
    predictor = dlib.shape_predictor(model_path)

    # Recursively gather images across all subfolders
    extensions = ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG')
    image_paths = []
    for ext in extensions:
        image_paths.extend(glob.glob(os.path.join(input_root, '**', ext), recursive=True))

    print(f"Discovered {len(image_paths)} images across dataset tree.")

    for img_path in tqdm(image_paths, desc="Aligning Faces"):
        rel_path = os.path.relpath(img_path, input_root)
        name_without_ext = os.path.splitext(rel_path)[0]
        out_path = os.path.join(output_root, f"{name_without_ext}.png")

        if os.path.exists(out_path):
            continue

        os.makedirs(os.path.dirname(out_path), exist_ok=True)

        try:
            img = dlib.load_rgb_image(img_path)
            dets = detector(img, 1)
            if len(dets) == 0:
                continue

            # Align using primary detected face
            landmarks = [(p.x, p.y) for p in predictor(img, dets[0]).parts()]
            image_align(img_path, out_path, landmarks, output_size=256)
        except Exception as e:
            print(f"Skipping {img_path}: {e}")

    print(f"Alignment completed. Files saved to: {output_root}")

In [8]:
# aligning the dataset if it hasn't been done before
if not os.path.exists(f"{ALIGNED_PATH}/{DATASET_NAME}"):
    align_dataset(
        input_root=DATASET_PATH, 
        output_root=f"{ALIGNED_PATH}/{DATASET_NAME}"
    )
else: 
    print("Aligned path exists, skipping the alignment process")

Aligned path exists, skipping the alignment process


In [9]:
# creating a guidleilne csv for morphing if it doesn't exist already
guideline_dir = f"/content/drive/MyDrive/Morph_guidelines/{DATASET_NAME}"
guideline_path = os.path.join(guideline_dir, "morph_guidelines.csv")

# do the guideline generation only if it doesn't exist already
if not os.path.exists(guideline_path):
    os.makedirs(guideline_dir, exist_ok=True)

    # get GPU if available, else fallback to CPU
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

    transform = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])

    # get the list of all aligned images
    aligned_dir = f"/content/drive/MyDrive/Datasets_aligned/{DATASET_NAME}"
    image_files = sorted([
        os.path.join(root, f) 
        for root, _, files in os.walk(aligned_dir) 
        for f in files if f.lower().endswith(('.png', '.jpg'))
    ])

    embeddings = []
    valid_files = []
    subject_ids = []

    # 1. Extract embeddings and parse subject IDs
    with torch.no_grad():
        for f in tqdm(image_files, desc="Extracting embeddings"):
            # Extracts ID from parent folder (e.g., '00491') or fallback to filename prefix
            subject_id = os.path.basename(os.path.dirname(f))
            if not subject_id or subject_id == os.path.basename(aligned_dir):
                subject_id = os.path.basename(f).split('_')[0]

            # Compute embedding
            img = Image.open(f).convert('RGB')
            tensor = transform(img).unsqueeze(0).to(device)
            emb = resnet(tensor).cpu().numpy().flatten()
            norm = np.linalg.norm(emb)
            if norm > 0:
                emb = emb / norm
                embeddings.append(emb)
                valid_files.append(f)
                subject_ids.append(subject_id)

    embeddings = np.array(embeddings)
    subject_ids = np.array(subject_ids)

    # 2. Compute Cosine Similarity Matrix
    similarity_matrix = np.dot(embeddings, embeddings.T)

    # 3. Mask out all images belonging to the same identity
    # Creates a boolean matrix where True indicates same subject
    same_identity_mask = subject_ids[:, None] == subject_ids[None, :]
    # Invalidate self and same-subject matches
    # basically set super low similarity score to the images beloginging to same identity
    similarity_matrix[same_identity_mask] = -1.0  

    # 4. Pair distinct identities greedily based on highest similarity
    morph_pairs = []
    # tracking used identites instead of single probes to get best, but one morph per subject
    used_subject_ids = set()

    for i in range(len(valid_files)):
        # skip if this subject already has morph suggested
        current_id = subject_ids[i]
        if current_id in used_subject_ids:
            continue

        # copy the scores for the current subject to avoid unnecesary iterations
        scores = similarity_matrix[i].copy()

        # Mask out all images belonging to already-paired identities
        for used_id in used_subject_ids:
            scores[subject_ids == used_id] = -1.0

        best_match_idx = int(np.argmax(scores))
        sim_score = scores[best_match_idx]
        best_id = subject_ids[best_match_idx]

        # Ensure valid inter-subject match meets threshold
        if sim_score >= 0.40 and best_id not in used_subject_ids:
                morph_pairs.append({
                    "id1": current_id,
                    "id2": best_id,
                    "img1": valid_files[i],
                    "img2": valid_files[best_match_idx],
                    "similarity": float(sim_score)
                })
                used_subject_ids.add(current_id)
                used_subject_ids.add(best_id)

    # 5. Save to CSV
    df_pairs = pd.DataFrame(morph_pairs)
    df_pairs.to_csv(guideline_path, index=False)
    print(f"Generated {len(df_pairs)} inter-subject morphing pairs -> {guideline_path}")

else:
    print(f"Morph guidelines already exist at {guideline_path}. Skipping generation.")

Morph guidelines already exist at /content/drive/MyDrive/Morph_guidelines/ferret_filtered_jpg_top3/morph_guidelines.csv. Skipping generation.


In [10]:
# implement the actual facemorp morphing here
aligned_dir = f"{ALIGNED_PATH}/{DATASET_NAME}"
suggestions_csv = f"/content/drive/MyDrive/Morph_guidelines/{DATASET_NAME}/morph_guidelines.csv"
output_root = f"/content/drive/MyDrive/Morphs/face_morpher/{DATASET_NAME}"
os.makedirs(output_root, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu" # not sure if there's a benefit for the face morpher to be on GPU

# importing the stuff necessary to use morpher as a module 
from facemorpher import aligner, blender, locator, warper
import cv2

def safe_bilinear_interpolate(img, coords):
    """Clamp warp coordinates so bilinear interpolation has a 2x2 neighborhood."""
    height, width = img.shape[:2]
    coords = coords.astype(np.float64, copy=True)
    coords[0] = np.clip(coords[0], 0, width - 1)
    coords[1] = np.clip(coords[1], 0, height - 1)

    int_coords = np.floor(coords).astype(np.int32)
    int_coords[0] = np.clip(int_coords[0], 0, width - 2)
    int_coords[1] = np.clip(int_coords[1], 0, height - 2)
    x0, y0 = int_coords
    dx = coords[0] - x0
    dy = coords[1] - y0

    q11 = img[y0, x0]
    q21 = img[y0, x0 + 1]
    q12 = img[y0 + 1, x0]
    q22 = img[y0 + 1, x0 + 1]

    btm = q21.T * dx + q11.T * (1 - dx)
    top = q22.T * dx + q12.T * (1 - dx)
    return (top * dy + btm * (1 - dy)).T

# The bundled warper assumes every sampled pixel has a valid right/bottom neighbor.
warper.bilinear_interpolate = safe_bilinear_interpolate

def load_aligned_image(path):
    image = cv2.imread(str(path))
    if image is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    h, w = image.shape[:2]

    # Pobieramy punkty dliba
    points = locator.face_points(image)
    if len(points) == 0:
        raise RuntimeError(f"No face found in: {path}")

    # 1. Zabezpieczenie punktów dliba przed wyjściem poza [0, w - 1] i [0, h - 1]
    points = np.array(points, dtype=np.float32)
    points[:, 0] = np.clip(points[:, 0], 0, w - 1)
    points[:, 1] = np.clip(points[:, 1], 0, h - 1)

    # 2. Bezpieczne punkty brzegowe kadru (indeksy maksymalne to w - 1 oraz h - 1)
    max_x = w - 1
    max_y = h - 1
    mid_x = w // 2
    mid_y = h // 2

    boundary_points = np.array([
        [0, 0], [mid_x, 0], [max_x, 0],
        [0, mid_y], [max_x, mid_y],
        [0, max_y], [mid_x, max_y], [max_x, max_y]
    ], dtype=np.float32)

    # Połączenie punktów
    all_points = np.vstack([points, boundary_points])

    return image, all_points

# ----------------------------
# read suggestions csv
# ----------------------------
df = pd.read_csv(suggestions_csv)

# using only images columns for morph generation
pairs = list(zip(df["img1"], df["img2"]))

# going over pairs the same way as in modiff 
for img1_path, img2_path in tqdm(pairs, desc="Generating morphs"):
    img1_name = os.path.splitext(os.path.basename(img1_path))[0]
    img2_name = os.path.splitext(os.path.basename(img2_path))[0]
    final_name = f"morphed_{img1_name}_and_{img2_name}.png"
    output_path = os.path.join(output_root, final_name)

    if os.path.exists(output_path):
        continue

    try:
        source_image, source_points = load_aligned_image(img1_path)
        destination_image, destination_points = load_aligned_image(img2_path)
        size = source_image.shape[:2]

        midpoint = locator.weighted_average_points(
            source_points, destination_points, percent=0.5
        )
        source_warp = warper.warp_image(source_image, source_points, midpoint, size)
        destination_warp = warper.warp_image(
            destination_image, destination_points, midpoint, size
        )
        result = blender.weighted_average(source_warp, destination_warp, percent=0.5)

        if not cv2.imwrite(str(output_path), result):
            raise IOError(f"Could not write output image: {output_path}")
    except Exception as e:
        print(f"Error generating morph for {img1_name} and {img2_name}: {e}")

Generating morphs:  10%|█         | 51/496 [00:54<06:13,  1.19it/s]

Error generating morph for 00054_931230_fa_a and 00168_931230_hl_a: No face found in: /content/drive/MyDrive/Datasets_aligned/ferret_filtered_jpg_top3/00168/00168_931230_hl_a.png


Generating morphs:  19%|█▊        | 92/496 [01:36<05:26,  1.24it/s]

Error generating morph for 00103_931230_fa_a and 00165_931230_hl_a: No face found in: /content/drive/MyDrive/Datasets_aligned/ferret_filtered_jpg_top3/00165/00165_931230_hl_a.png


Generating morphs: 100%|██████████| 496/496 [08:26<00:00,  1.02s/it]
